## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [1]:
## add your code here
import sys

MAXN = 1024

n = 0
A = 0
B = 0
L = 0

cur = []
pos_of = []
ans = []


def refresh_pos():
    global pos_of
    for i in range(n):
        pos_of[cur[i]] = i


def use_swap_magic():
    """
    交换魔法：
    交换当前所有石板中数字 A 和 B 的位置
    输出操作编号 0
    """
    ans.append(0)

    for i in range(n):
        if cur[i] == A:
            cur[i] = B
        elif cur[i] == B:
            cur[i] = A

    refresh_pos()


def use_add(x):
    """
    加法魔法：
    每个数字加 x，结果对 n 取模
    输出操作 2 x
    """
    x %= n
    if x == 0:
        return

    ans.append(x)

    for i in range(n):
        cur[i] = (cur[i] + x) % n

    refresh_pos()


def use_xor(x):
    """
    异或魔法：
    每个数字异或 x
    输出操作 1 x
    """
    if x == 0:
        return

    ans.append(-x)

    for i in range(n):
        cur[i] ^= x

    refresh_pos()


def calc_pair(x, y):
    """
    将一对数 x, y 转换成标准形，
    方便借助固定交换 A, B 来模拟交换 x, y
    """
    delta = (y - x + n - L + n) % n

    px = 0
    py = 0

    step = n // 2
    while step >= 2 * L:
        if delta >= step:
            delta -= step
            py += step // 2
        else:
            px += step // 2
        step //= 2

    px += n // 2
    px += x & (L - 1)

    py += x & (L - 1)

    return px, py


def swap_value(x, y):
    """
    模拟交换当前排列中的数字 x 和 y
    """
    if x == y:
        return

    # 如果 x 和 y 在同类块中，需要借助一个中转点交换
    if (x // L) % 2 == (y // L) % 2:
        if (x // L) % 2 == 0:
            mid = (x & (L - 1)) + L
        else:
            mid = x & (L - 1)

        swap_value(x, mid)
        swap_value(y, mid)
        swap_value(x, mid)
        return

    pA, pB = calc_pair(A, B)
    px, py = calc_pair(x, y)

    use_add((px - x + n) % n)
    use_xor(px ^ pA)
    use_add((A - pA + n) % n)

    use_swap_magic()

    use_add((pA - A + n) % n)
    use_xor(px ^ pA)
    use_add((x - px + n) % n)


class PermSolver:
    def __init__(self, length=0):
        self.length = length
        self.p = [0] * MAXN
        self.seq = []

    def solve(self):
        """
        递归处理低位排列。
        seq 中：
        正数表示加法操作
        负数表示异或操作
        """

        seen = [0] * self.length

        for i in range(self.length):
            if self.p[i] < 0 or self.p[i] >= self.length or seen[self.p[i]]:
                return False
            seen[self.p[i]] = 1

        if self.length == 1:
            return True

        even_part = PermSolver(self.length // 2)
        odd_part = PermSolver(self.length // 2)

        for i in range(self.length // 2):
            even_part.p[i] = self.p[i * 2] // 2
            odd_part.p[i] = self.p[i * 2 + 1] // 2

        if not even_part.solve():
            return False

        if not odd_part.solve():
            return False

        if self.p[0] & 1:
            if self.length == 2:
                self.seq.append(1)
            else:
                self.seq.append(-1)

        xor_even = 0

        for x in even_part.seq:
            if x > 0:
                self.seq.append(-1)
                self.seq.append(1)
            else:
                self.seq.append(x * 2)
                xor_even ^= (-x * 2)

        if xor_even:
            self.seq.append(-xor_even)

        xor_odd = 0

        for x in odd_part.seq:
            if x > 0:
                self.seq.append(1)
                self.seq.append(-1)
            else:
                self.seq.append(x * 2)
                xor_odd ^= (-x * 2)

        if (xor_odd & (self.length // 2)) != (xor_even & (self.length // 2)):
            return False

        if xor_even >= self.length // 2:
            xor_even -= self.length // 2

        if xor_odd >= self.length // 2:
            xor_odd -= self.length // 2

        if xor_even != xor_odd:
            return False

        compact = []

        for x in self.seq:
            if compact and x < 0 and compact[-1] < 0:
                compact[-1] = -((-compact[-1]) ^ (-x))

                if compact[-1] == 0:
                    compact.pop()
            elif x != 0:
                compact.append(x)

        self.seq = compact
        return True


def main():
    global n, A, B, L, cur, pos_of, ans

    data = list(map(int, sys.stdin.buffer.read().split()))

    if not data:
        return

    idx = 0

    n = data[idx]
    idx += 1

    A = data[idx]
    B = data[idx + 1]
    idx += 2

    cur = data[idx:idx + n]
    idx += n

    pos_of = [0] * n
    ans = []

    refresh_pos()

    # L 是 A 和 B 差值的 lowbit
    L = (A - B + n) % n
    L = L & -L

    if L == 0:
        L = n

    # 第一步：先用加法和异或处理低 L 位
    if L > 1:
        solver = PermSolver(L)

        for i in range(L):
            solver.p[i] = cur[i] & (L - 1)

        if not solver.solve():
            print(-1)
            return

        for x in solver.seq:
            if x > 0:
                use_add(x)
            else:
                use_xor(-x)

    # 第二步：按照 mod L 分组检查并排序
    for r in range(L):
        values = []

        for i in range(r, n, L):
            values.append(cur[i])

        values.sort()

        idx = 0
        for i in range(r, n, L):
            if values[idx] != i:
                print(-1)
                return
            idx += 1

        for i in range(r, n, L):
            if cur[i] != i:
                swap_value(i, cur[i])

    # 最终检查
    for i in range(n):
        if cur[i] != i:
            print(-1)
            return

    output = []
    output.append(str(len(ans)))

    for op in ans:
        if op == 0:
            output.append("0")
        elif op < 0:
            output.append(f"1 {-op}")
        else:
            output.append(f"2 {op}")

    sys.stdout.write("\n".join(output))


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
## add your code here
import sys

def main():
    data = sys.stdin.read().split()
    if not data:
        return

    idx = 0
    ans = []

    while idx < len(data):
        N = int(data[idx])
        L = int(data[idx + 1])
        Maxn = int(data[idx + 2])
        S = int(data[idx + 3])
        idx += 4

        stations = {}
        for _ in range(N):
            p = int(data[idx])
            c = int(data[idx + 1])
            idx += 2

            # 同一位置可能有多个补给站，只保留最便宜的即可
            if p not in stations or c < stations[p]:
                stations[p] = c

        # 起点0，终点L
        positions = [0] + sorted(stations.keys()) + [L]
        costs = [0] + [stations[p] for p in sorted(stations.keys())] + [0]

        m = len(positions)
        INF = 10**18
        dp = [INF] * m
        dp[0] = 0  # 起点出发，初始满体力，不花钱

        # dp[i] = 到达 positions[i]，并把这里当作“当前满体力出发点”时的最小花费
        # 对于终点L，不需要补给，所以 cost[L] = 0
        for i in range(m):
            if dp[i] == INF:
                continue
            for j in range(i + 1, m):
                if positions[j] - positions[i] > Maxn:
                    break
                dp[j] = min(dp[j], dp[i] + costs[j])

        ans.append("Yes" if dp[-1] <= S else "No")

    print("\n".join(ans))

if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
## add your code here
import sys


def manacher(s):
    n = len(s)
    d1 = [0] * n
    d2 = [0] * n

    # 奇数长度回文
    l, r = 0, -1
    for i in range(n):
        if i > r:
            k = 1
        else:
            k = min(d1[l + r - i], r - i + 1)

        while i - k >= 0 and i + k < n and s[i - k] == s[i + k]:
            k += 1

        d1[i] = k

        if i + k - 1 > r:
            l = i - k + 1
            r = i + k - 1

    # 偶数长度回文
    l, r = 0, -1
    for i in range(n):
        if i > r:
            k = 0
        else:
            k = min(d2[l + r - i + 1], r - i + 1)

        while i - k - 1 >= 0 and i + k < n and s[i - k - 1] == s[i + k]:
            k += 1

        d2[i] = k

        if i + k - 1 > r:
            l = i - k
            r = i + k - 1

    return d1, d2


class DoubleHash:
    def __init__(self, s, base, mod1, mod2, pow1, pow2):
        self.s = s
        self.mod1 = mod1
        self.mod2 = mod2
        self.pow1 = pow1
        self.pow2 = pow2

        n = len(s)
        self.h1 = [0] * (n + 1)
        self.h2 = [0] * (n + 1)

        for i, ch in enumerate(s):
            v = ch + 1
            self.h1[i + 1] = (self.h1[i] * base + v) % mod1
            self.h2[i + 1] = (self.h2[i] * base + v) % mod2

    def get_hash(self, l, r):
        x1 = (self.h1[r] - self.h1[l] * self.pow1[r - l]) % self.mod1
        x2 = (self.h2[r] - self.h2[l] * self.pow2[r - l]) % self.mod2
        return x1, x2


def main():
    data = sys.stdin.buffer.read().split()
    if not data:
        return

    n = int(data[0])
    A = data[1]
    B = data[2]

    RA = A[::-1]

    d1A, d2A = manacher(A)
    d1B, d2B = manacher(B)

    MOD1 = 1000000007
    MOD2 = 1000000009
    BASE = 911382323

    pow1 = [1] * (n + 1)
    pow2 = [1] * (n + 1)

    for i in range(n):
        pow1[i + 1] = pow1[i] * BASE % MOD1
        pow2[i + 1] = pow2[i] * BASE % MOD2

    hash_RA = DoubleHash(RA, BASE, MOD1, MOD2, pow1, pow2)
    hash_B = DoubleHash(B, BASE, MOD1, MOD2, pow1, pow2)

    def lcp_RAB(posRA, posB):
        """
        求 RA[posRA:] 和 B[posB:] 的最长公共前缀长度
        """
        if posRA < 0 or posRA >= n or posB < 0 or posB >= n:
            return 0

        left = 0
        right = min(n - posRA, n - posB)

        while left < right:
            mid = (left + right + 1) // 2

            if hash_RA.get_hash(posRA, posRA + mid) == hash_B.get_hash(posB, posB + mid):
                left = mid
            else:
                right = mid - 1

        return left

    ans = 0

    # 情况 1：回文中心在 A 中，奇数长度回文
    for i in range(n):
        R = d1A[i] - 1

        left = i - R
        right = i + R

        if left == 0:
            q = 0
        else:
            q = lcp_RAB(n - left, right)

        ans = max(ans, 2 * R + 1 + 2 * q)

    # 情况 2：回文中心在 A 中，偶数长度回文
    for i in range(1, n + 1):
        if i < n:
            R = d2A[i]
        else:
            R = 0

        left = i - R
        right = i + R - 1

        if left == 0:
            q = 0
        else:
            q = lcp_RAB(n - left, right)

        ans = max(ans, 2 * R + 2 * q)

    # 情况 3：回文中心在 B 中，奇数长度回文
    for i in range(n):
        R = d1B[i] - 1

        left = i - R
        right = i + R

        if right + 1 >= n:
            q = 0
        else:
            q = lcp_RAB(n - 1 - left, right + 1)

        ans = max(ans, 2 * R + 1 + 2 * q)

    # 情况 4：回文中心在 B 中，偶数长度回文
    for i in range(n):
        R = d2B[i]

        left = i - R
        right = i + R - 1

        if right + 1 >= n:
            q = 0
        else:
            q = lcp_RAB(n - 1 - left, right + 1)

        ans = max(ans, 2 * R + 2 * q)

    print(ans)


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
## add your code here
import sys


class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 2)

    def add(self, idx, val):
        while idx <= self.n:
            self.tree[idx] += val
            idx += idx & -idx

    def sum(self, idx):
        if idx <= 0:
            return 0
        res = 0
        while idx > 0:
            res += self.tree[idx]
            idx -= idx & -idx
        return res

    def kth(self, k):
        """
        找到第 k 个值为 1 的位置
        """
        idx = 0
        bit = 1 << (self.n.bit_length() - 1)

        while bit:
            nxt = idx + bit
            if nxt <= self.n and self.tree[nxt] < k:
                idx = nxt
                k -= self.tree[nxt]
            bit >>= 1

        return idx + 1

    def lower_bound(self, pos):
        """
        找到当前还没被使用的 ? 中，第一个 >= pos 的行号
        如果不存在，返回 -1
        """
        if pos < 1:
            pos = 1

        before = self.sum(pos - 1)
        total = self.sum(self.n)

        if total == before:
            return -1

        return self.kth(before + 1)


def solve():
    data = sys.stdin.buffer.read().split()
    idx = 0
    out = []

    state = []
    last_pos = []
    modified = []

    while idx < len(data):
        m = int(data[idx])
        idx += 1

        if m == 0:
            out.append("-1")
            continue

        bit = Fenwick(m)

        error_line = -1
        modified.clear()

        for i in range(1, m + 1):
            op = data[idx]
            idx += 1

            x = -1

            if op == b'I' or op == b'O':
                x = int(data[idx])
                idx += 1

                if x >= len(state):
                    new_size = max(len(state) * 2, x + 10000)
                    if new_size == 0:
                        new_size = x + 10000

                    state.extend([0] * (new_size - len(state)))
                    last_pos.extend([0] * (new_size - len(last_pos)))

            # 如果已经发现错误，后面只读取，不再处理
            if error_line != -1:
                continue

            if op == b'?':
                bit.add(i, 1)

            elif op == b'I':
                if state[x] == 1:
                    # 已经有 x，又购买 x
                    # 需要找一个 ? 当作 O x
                    pos = bit.lower_bound(last_pos[x])

                    if pos != -1 and pos < i:
                        bit.add(pos, -1)
                    else:
                        error_line = i

                state[x] = 1
                last_pos[x] = i
                modified.append(x)

            elif op == b'O':
                if state[x] == 0:
                    # 没有 x，却使用 x
                    # 需要找一个 ? 当作 I x
                    pos = bit.lower_bound(last_pos[x])

                    if pos != -1 and pos < i:
                        bit.add(pos, -1)
                    else:
                        error_line = i

                state[x] = 0
                last_pos[x] = i
                modified.append(x)

        out.append(str(error_line))

        # 恢复本组数据修改过的位置
        for x in modified:
            state[x] = 0
            last_pos[x] = 0

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    solve()

## E 任意点

In [ ]:
## add your code here
import sys
from collections import defaultdict, deque

def main():
    data = sys.stdin.read().split()
    if not data:
        return
    n = int(data[0])
    points = []
    idx = 1
    for i in range(n):
        x = int(data[idx]); y = int(data[idx+1]); idx += 2
        points.append((x, y))
    
    # 如果只有一个点，已经连通
    if n <= 1:
        print(0)
        return
    
    # 建图：邻接表
    graph = [[] for _ in range(n)]
    for i in range(n):
        for j in range(i+1, n):
            x1, y1 = points[i]
            x2, y2 = points[j]
            if x1 == x2 or y1 == y2:
                graph[i].append(j)
                graph[j].append(i)
    
    # BFS/DFS 求连通分量数
    visited = [False] * n
    components = 0
    for i in range(n):
        if not visited[i]:
            components += 1
            stack = [i]
            visited[i] = True
            while stack:
                u = stack.pop()
                for v in graph[u]:
                    if not visited[v]:
                        visited[v] = True
                        stack.append(v)
    
    print(components - 1)

if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
## add your code here
import sys


class Block:
    __slots__ = ("off", "lit")

    def __init__(self, off, lit):
        self.off = off
        self.lit = lit


class Piece:
    __slots__ = ("length", "blocks", "anchor")

    def __init__(self):
        self.length = 0
        self.blocks = []
        self.anchor = -1


def compress_stars(p):
    res = []
    prev_star = False

    for c in p:
        if c == ord('*'):
            if not prev_star:
                res.append(c)
            prev_star = True
        else:
            res.append(c)
            prev_star = False

    return bytes(res)


def build_piece(part):
    pc = Piece()
    pc.length = len(part)

    n = len(part)
    i = 0

    while i < n:
        if part[i] == ord('?'):
            i += 1
            continue

        j = i
        while j < n and part[j] != ord('?'):
            j += 1

        pc.blocks.append(Block(i, part[i:j]))
        i = j

    best = -1
    best_len = -1

    for idx, block in enumerate(pc.blocks):
        cur_len = len(block.lit)
        if cur_len > best_len:
            best_len = cur_len
            best = idx

    pc.anchor = best
    return pc


def check_piece_at(s, pos, pc):
    """
    判断 piece 是否能从 s[pos] 开始匹配
    """
    if pos < 0 or pos + pc.length > len(s):
        return False

    for block in pc.blocks:
        start = pos + block.off
        end = start + len(block.lit)

        if s[start:end] != block.lit:
            return False

    return True


def find_piece(s, left, right, pc):
    """
    在 s[left:right] 中找最早能匹配 pc 的起点
    """
    if pc.length > right - left:
        return -1

    # 这一段全是 ?，直接放在最早位置
    if pc.anchor == -1:
        return left

    anchor_block = pc.blocks[pc.anchor]

    min_anchor_pos = left + anchor_block.off
    max_anchor_pos = right - pc.length + anchor_block.off

    start = min_anchor_pos

    while start <= max_anchor_pos:
        pos = s.find(anchor_block.lit, start)

        if pos == -1 or pos > max_anchor_pos:
            return -1

        candidate = pos - anchor_block.off

        if check_piece_at(s, candidate, pc):
            return candidate

        start = pos + 1

    return -1


def main():
    data = sys.stdin.buffer.read().split()

    if not data:
        return

    pat = data[0]
    n = int(data[1])

    pat = compress_stars(pat)

    min_need = 0
    star_count = 0

    for c in pat:
        if c == ord('*'):
            star_count += 1
        else:
            min_need += 1

    parts = pat.split(b'*')

    lead_star = len(pat) > 0 and pat[0] == ord('*')
    tail_star = len(pat) > 0 and pat[-1] == ord('*')

    pieces = [build_piece(part) for part in parts]

    out = []
    strings = data[2:2 + n]

    for s in strings:
        slen = len(s)

        if slen < min_need:
            out.append("NO")
            continue

        # 没有 *，必须整串匹配
        if star_count == 0:
            if slen != len(pat):
                out.append("NO")
                continue

            ok = True

            for i in range(slen):
                if pat[i] != ord('?') and pat[i] != s[i]:
                    ok = False
                    break

            out.append("YES" if ok else "NO")
            continue

        left = 0
        right = slen
        ok = True

        left_piece = 0
        right_piece = len(pieces) - 1

        # 前缀锚定：模式不是以 * 开头时，第一段必须从 s[0] 开始匹配
        if not lead_star:
            pc = pieces[0]

            if not check_piece_at(s, 0, pc):
                ok = False
            else:
                left += pc.length

            left_piece = 1

        # 后缀锚定：模式不是以 * 结尾时，最后一段必须贴着字符串末尾匹配
        if ok and not tail_star:
            pc = pieces[-1]
            pos = slen - pc.length

            if not check_piece_at(s, pos, pc):
                ok = False
            else:
                right -= pc.length

            right_piece = len(pieces) - 2

        if not ok or left > right:
            out.append("NO")
            continue

        # 中间段按顺序找最早匹配位置
        for i in range(left_piece, right_piece + 1):
            pc = pieces[i]
            pos = find_piece(s, left, right, pc)

            if pos == -1:
                ok = False
                break

            left = pos + pc.length

        out.append("YES" if ok else "NO")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
import sys

def parse_ops(line: str):
    # 兼容 "AB AC BA BC CA CB" 或 "ABACBABC..." 这两种形式
    letters = [ch for ch in line if ch in "ABC"]
    ops = []
    for i in range(0, len(letters), 2):
        ops.append((letters[i], letters[i + 1]))
    return ops

def simulate(n, ops):
    A = list(range(n, 0, -1))
    B = []
    C = []
    stacks = {'A': A, 'B': B, 'C': C}

    last_disk = 0
    steps = 0

    while len(B) < n and len(C) < n:
        for src, dst in ops:
            s = stacks[src]
            d = stacks[dst]

            if not s:
                continue

            disk = s[-1]

            if disk == last_disk:
                continue

            if not d or d[-1] > disk:
                s.pop()
                d.append(disk)
                last_disk = disk
                steps += 1
                break

    return steps

def main():
    data = sys.stdin.read().splitlines()
    if not data:
        return

    n = int(data[0].strip())
    ops = parse_ops(data[1].strip())

    # 小数据直接模拟
    if n <= 3:
        print(simulate(n, ops))
        return

    # 只模拟 3 个盘子来判型
    s3 = simulate(3, ops)

    if s3 == 7:
        # 2^n - 1
        print((1 << n) - 1)
    elif s3 == 9:
        # 3^(n-1)
        print(pow(3, n - 1))
    else:
        # s3 == 17
        # 2 * 3^(n-1) - 1
        print(2 * pow(3, n - 1) - 1)

if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
## add your code here
import sys

def knight_distance(dx, dy):
    dx, dy = abs(dx), abs(dy)
    if dx < dy:
        dx, dy = dy, dx
    
    # 特殊情况
    if dx == 0 and dy == 0:
        return 0
    if dx == 1 and dy == 0:
        return 3
    if dx == 2 and dy == 2:
        return 4
    if dx == 2 and dy == 0:
        return 2
    if dx == 3 and dy == 0:
        return 3
    if dx == 3 and dy == 1:
        return 2
    if dx == 4 and dy == 0:
        return 2
    if dx == 4 and dy == 2:
        return 2

    # 通用公式
    steps = max((dx + 1) // 2, (dx + dy + 2) // 3)
    # 调整奇偶性：马每步改变 (x+y) 的奇偶性（因为 Δx+Δy = ±1±2 = ±3 或 ±2±1 = ±3，都是奇数）
    if (steps % 2) != ((dx + dy) % 2):
        steps += 1

    return steps

def main():
    data = sys.stdin.read().split()
    if not data:
        return
    xp = int(data[0]); yp = int(data[1]); xs = int(data[2]); ys = int(data[3])
    
    result = knight_distance(xs - xp, ys - yp)
    print(result)

if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
## add your code here
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        # 添加哨兵0，确保最后能清空栈
        heights = heights + [0]
        stack = []
        max_area = 0
        
        for i in range(len(heights)):
            # 当前高度小于栈顶时，计算以栈顶为高的矩形面积
            while stack and heights[i] < heights[stack[-1]]:
                # 弹出栈顶，作为矩形的高度
                h = heights[stack.pop()]
                
                # 计算宽度
                # 左边界是新的栈顶（第一个比当前高度小的位置），如果栈空则为-1
                left_bound = stack[-1] if stack else -1
                width = i - left_bound - 1
                
                # 更新最大面积
                max_area = max(max_area, h * width)
            
            # 当前索引入栈
            stack.append(i)
        
        return max_area

## J 消防局的设立

In [ ]:
## add your code here
import sys
from collections import deque

def main():
    data = sys.stdin.read().strip().split()
    if not data:
        return

    n = int(data[0])

    graph = [[] for _ in range(n + 1)]
    for i in range(1, n):
        p = int(data[i])
        u = i + 1
        graph[p].append(u)
        graph[u].append(p)

    # BFS 求父节点和深度
    depth = [-1] * (n + 1)
    parent = [0] * (n + 1)
    q = deque([1])
    depth[1] = 0

    while q:
        u = q.popleft()
        for v in graph[u]:
            if depth[v] == -1:
                depth[v] = depth[u] + 1
                parent[v] = u
                q.append(v)

    # 按深度从大到小处理
    nodes = list(range(1, n + 1))
    nodes.sort(key=lambda x: depth[x], reverse=True)

    covered = [False] * (n + 1)
    ans = 0

    def mark_covered(start):
        # 本次 BFS 的访问判重，不能和 covered 混用
        vis = set([start])
        q = deque([(start, 0)])

        while q:
            u, d = q.popleft()
            covered[u] = True
            if d == 2:
                continue
            for v in graph[u]:
                if v not in vis:
                    vis.add(v)
                    q.append((v, d + 1))

    for u in nodes:
        if covered[u]:
            continue

        # 往上跳 2 层，放在祖父处
        place = u
        for _ in range(2):
            if parent[place] != 0:
                place = parent[place]

        ans += 1
        mark_covered(place)

    print(ans)

if __name__ == "__main__":
    main()